# EPC 03

In [1]:
import pandas as pd
import random
import math

In [2]:
class PerceptronMultiLayer:
    def __init__(self, layers, learning_rate=0.1, momentum=0.0, beta=0.5, seed=None):
        self.layers = layers
        self.eta = learning_rate
        self.alpha = momentum          
        self.beta = beta
        self.rng = random.Random(seed)
        # W[l][j][i]: peso do neurônio j da camada l+1 vindo da entrada i
        # i=0 é o bias ligado a -1
        self.W = [[[self.rng.uniform(0, 1) for _ in range(layers[l] + 1)]
                   for _ in range(layers[l + 1])]
                  for l in range(len(layers) - 1)]
        self.dW_prev = [[[0.0] * (layers[l] + 1) for _ in range(layers[l + 1])]
                        for l in range(len(layers) - 1)]

    def g(self, u):
        return 1.0 / (1.0 + math.exp(-self.beta * u))

    def forward(self, x):
        acts = [list(x)]
        for W in self.W:
            xb = [-1.0] + acts[-1]
            saida = []
            for pesos in W:
                u = 0.0
                for w, a in zip(pesos, xb):
                    u += w * a
                saida.append(self.g(u))
            acts.append(saida)
        return acts

    def backward(self, acts, y):
        beta = self.beta
        # delta da saída: (y - o) * g'(u), com g'(u) = beta * g * (1 - g)
        delta = [(y[j] - o) * beta * o * (1 - o) for j, o in enumerate(acts[-1])]
        for l in range(len(self.W) - 1, -1, -1):
            xb = [-1.0] + acts[l]
            W, dW_prev = self.W[l], self.dW_prev[l]
            if l > 0:  
                prox = []
                for i, a in enumerate(acts[l]):
                    erro = 0.0
                    for j in range(len(delta)):
                        erro += W[j][i + 1] * delta[j]
                    prox.append(erro * beta * a * (1 - a))
            # atuualização de pesos: Δw(t) = η·δ·x + αΔw(t-1)
            for j in range(len(W)):
                dj, pj, dpj = delta[j], W[j], dW_prev[j]
                for i in range(len(pj)):
                    d = self.eta * dj * xb[i] + self.alpha * dpj[i]
                    pj[i] += d
                    dpj[i] = d
            if l > 0:
                delta = prox

    def predict(self, x):
        return self.forward(x)[-1]

    def eqm(self, X, Y):
        total = 0.0
        for x, y in zip(X, Y):
            saida = self.predict(x)
            for j in range(len(saida)):
                total += (y[j] - saida[j]) ** 2
        return total / (2 * len(X))

    def fit(self, X, Y, epsilon=1e-6, max_epochs=3000):
        historico, eqm_ant = [], float("inf")
        idx = list(range(len(X)))
        for epoca in range(1, max_epochs + 1):
            self.rng.shuffle(idx)                  
            total = 0.0
            for i in idx:
                acts = self.forward(X[i])
                for j in range(len(acts[-1])):
                    total += (Y[i][j] - acts[-1][j]) ** 2
                self.backward(acts, Y[i])
            eqm_atual = total / (2 * len(X))
            historico.append(eqm_atual)
            if abs(eqm_atual - eqm_ant) <= epsilon:   
                break
            eqm_ant = eqm_atual
        return epoca, historico

In [3]:
input_data = []
output_data= []

with open('data/iris.dat', 'r', encoding='utf-8') as file:

    for linha in file:
        linha = linha.strip()

        if linha == "@data":
            break

    for linha in file:
        dados = linha.split(",")


        input_data.append([
            float(dados[0]),
            float(dados[1]),
            float(dados[2]),
            float(dados[3]),

        ])
        output_config = [
            [0, 0],
            [0, 1],
            [1, 0]
        ]
        output_data_raw = dados[4].strip()
        match output_data_raw: 
            case 'Iris-setosa':
                output_data.append(output_config[0])
            case 'Iris-versicolor':
                output_data.append(output_config[1])
            case _:
                output_data.append(output_config[2])

print("Inputs:", input_data)
print("Outputs:", output_data)

Inputs: [[5.1, 3.5, 1.4, 0.2], [4.9, 3.0, 1.4, 0.2], [4.6, 3.1, 1.5, 0.2], [5.0, 3.6, 1.4, 0.2], [5.4, 3.9, 1.7, 0.4], [4.6, 3.4, 1.4, 0.3], [5.0, 3.4, 1.5, 0.2], [4.4, 2.9, 1.4, 0.2], [5.4, 3.7, 1.5, 0.2], [4.8, 3.4, 1.6, 0.2], [4.8, 3.0, 1.4, 0.1], [4.3, 3.0, 1.1, 0.1], [5.7, 4.4, 1.5, 0.4], [5.4, 3.9, 1.3, 0.4], [5.1, 3.5, 1.4, 0.3], [5.7, 3.8, 1.7, 0.3], [5.1, 3.8, 1.5, 0.3], [5.4, 3.4, 1.7, 0.2], [5.1, 3.7, 1.5, 0.4], [4.6, 3.6, 1.0, 0.2], [5.1, 3.3, 1.7, 0.5], [4.8, 3.4, 1.9, 0.2], [5.0, 3.0, 1.6, 0.2], [5.0, 3.4, 1.6, 0.4], [5.2, 3.5, 1.5, 0.2], [5.2, 3.4, 1.4, 0.2], [4.7, 3.2, 1.6, 0.2], [4.8, 3.1, 1.6, 0.2], [5.4, 3.4, 1.5, 0.4], [5.2, 4.1, 1.5, 0.1], [5.5, 4.2, 1.4, 0.2], [4.9, 3.1, 1.5, 0.1], [5.0, 3.2, 1.2, 0.2], [5.5, 3.5, 1.3, 0.2], [4.9, 3.1, 1.5, 0.1], [4.4, 3.0, 1.3, 0.2], [5.1, 3.4, 1.5, 0.2], [5.0, 3.5, 1.3, 0.3], [4.5, 2.3, 1.3, 0.3], [5.1, 3.8, 1.9, 0.4], [4.8, 3.0, 1.4, 0.3], [5.1, 3.8, 1.6, 0.2], [4.6, 3.2, 1.4, 0.2], [5.3, 3.7, 1.5, 0.2], [5.0, 3.3, 1.4, 0.2], [

In [4]:
#to-do: train_test_split
def train_test_split(X, y, test_size=0.2, random_state = 42):

    if len(X) != len(y):
        raise ValueError("X and y must be the same length")

    generator = random.Random(random_state)

    indexs = list(range(len(X)))
    generator.shuffle(indexs)

    tot_items = len(X)
    test_size = int(tot_items * test_size)

    #pega até o tamanho de teste, depois o que sobrou é o treinamento
    indexs_test = indexs[:test_size]
    indexs_train = indexs[test_size:]

    X_train = [X[i] for i in indexs_train]
    X_test = [X[i] for i in indexs_test]
    y_train = [y[i] for i in indexs_train]
    y_test = [y[i] for i in indexs_test]

    return X_train, X_test, y_train, y_test

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    input_data, output_data, test_size=0.1, random_state=42
)
print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")

Treino: 135 | Teste: 15


In [6]:
def pos_process(saida):
    return [1 if y >= 0.5 else 0 for y in saida]

def acuracia(model, X, Y):
    acertos = sum(pos_process(model.predict(x)) == list(y) for x, y in zip(X, Y))
    return acertos / len(X)

def media(l):
    return sum(l) / len(l)

def desvio(l):
    m = media(l)
    return math.sqrt(sum((v - m) ** 2 for v in l) / len(l)) 

## 2. Relatório

#### **Questão 1**

In [7]:
def stratified_folds(y, k, seed=42):
    rng = random.Random(seed)
    por_classe = {}
    for i, r in enumerate(y):
        por_classe.setdefault(tuple(r), []).append(i)
    folds, cont = [[] for _ in range(k)], 0
    for r in sorted(por_classe):
        idx = por_classe[r]
        rng.shuffle(idx)
        for i in idx:
            folds[cont % k].append(i)
            cont += 1
    return folds

In [8]:
def cross_validation(X, y, layers, k=10, learning_rate=0.1, momentum=0.0,
                     epsilon=1e-6, max_epochs=3000, seed=42):
    folds = stratified_folds(y, k, seed)
    res = {"eqm": [], "epocas": [], "acc_val": [], "modelos": []}

    for f in range(k):
        val_idx = folds[f]                                          #validação (10%)
        tr_idx = [i for g in range(k) if g != f for i in folds[g]]  #treino (90%)

        Xtr = [X[i] for i in tr_idx]; ytr = [y[i] for i in tr_idx]
        Xv  = [X[i] for i in val_idx]; yv  = [y[i] for i in val_idx]

        m = PerceptronMultiLayer(layers, learning_rate, momentum, seed=seed + f)
        epocas, _ = m.fit(Xtr, ytr, epsilon, max_epochs)

        res["epocas"].append(epocas)
        res["eqm"].append(m.eqm(Xv, yv))
        res["acc_val"].append(acuracia(m, Xv, yv) * 100)
        res["modelos"].append(m)
    return res

In [9]:
topologias = {"4-5-2": [4, 5, 2], "4-10-2": [4, 10, 2],
              "4-15-2": [4, 15, 2], "4-20-2": [4, 20, 2]}

def rodar_experimento(momentum, max_epochs=3000):
    brutos, linhas = {}, []
    for nome, cfg in topologias.items():
        print(f"Treinando {nome} (momentum={momentum})...")
        r = cross_validation(X_train, y_train, cfg, k=10,
                             momentum=momentum, max_epochs=max_epochs)
        brutos[nome] = r
        linhas.append({
            "Topologia": nome,
            "EQM": f"{media(r['eqm']):.4f} ± {desvio(r['eqm']):.4f}",
            "Acurácia (%)": f"{media(r['acc_val']):.2f} ± {desvio(r['acc_val']):.2f}",
            "Épocas": f"{media(r['epocas']):.0f} ± {desvio(r['epocas']):.0f}",
            "_acc": media(r["acc_val"]), "_eqm": media(r["eqm"]),
        })
    df = pd.DataFrame(linhas)
    melhor = df.sort_values(["_acc", "_eqm"], ascending=[False, True]).iloc[0]["Topologia"]
    return df.drop(columns=["_acc", "_eqm"]), brutos, melhor

#### **Questão 2**

In [10]:
df_padrao, brutos_padrao, melhor_padrao = rodar_experimento(momentum=0.0)
 
print("Melhor:", melhor_padrao)
df_padrao

Treinando 4-5-2 (momentum=0.0)...
Treinando 4-10-2 (momentum=0.0)...
Treinando 4-15-2 (momentum=0.0)...
Treinando 4-20-2 (momentum=0.0)...
Melhor: 4-10-2


,Topologia,EQM,Acurácia (%),Épocas
0,4-5-2,0.0150 ± 0.0195,99.29 ± 2.14,795 ± 285
1,4-10-2,0.0138 ± 0.0198,99.29 ± 2.14,787 ± 360
2,4-15-2,0.0164 ± 0.0197,99.29 ± 2.14,563 ± 272
3,4-20-2,0.0209 ± 0.0308,97.86 ± 4.57,850 ± 419


#### **Questão 3**

In [11]:
df_mom, brutos_mom, melhor_mom = rodar_experimento(momentum=0.6)

print("Melhor:", melhor_mom)
df_mom

Treinando 4-5-2 (momentum=0.6)...
Treinando 4-10-2 (momentum=0.6)...
Treinando 4-15-2 (momentum=0.6)...
Treinando 4-20-2 (momentum=0.6)...
Melhor: 4-15-2


,Topologia,EQM,Acurácia (%),Épocas
0,4-5-2,0.0105 ± 0.0202,99.29 ± 2.14,784 ± 270
1,4-10-2,0.0105 ± 0.0204,99.29 ± 2.14,592 ± 218
2,4-15-2,0.0096 ± 0.0204,99.29 ± 2.14,708 ± 347
3,4-20-2,0.0111 ± 0.0205,99.29 ± 2.14,956 ± 480


#### **Questão 4**

In [12]:
def linha_teste(nome_algo, melhor, brutos):
    accs = [acuracia(m, X_test, y_test) * 100 for m in brutos[melhor]["modelos"]]
    return {"Algoritmo de treinamento": nome_algo, "Topologia": melhor,
            "Acurácia (%)": f"{media(accs):.2f} ± {desvio(accs):.2f}"}

df_teste = pd.DataFrame([
    linha_teste("Backpropagation padrão", melhor_padrao, brutos_padrao),
    linha_teste("Backpropagation com momentum", melhor_mom, brutos_mom),
])
df_teste

,Algoritmo de treinamento,Topologia,Acurácia (%)
0,Backpropagation padrão,4-10-2,81.33 ± 4.99
1,Backpropagation com momentum,4-15-2,80.00 ± 4.22


#### **Questão 5**

Underfitting ocorre quando o modelo não conseguiu aprender os padrões do dados. Isso pode acontecer devido a uma topologia muito simples, sendo insuficiente para resolver o problema e, também, por treinamento insuficiente. Sobre a detecção: quando o modelo apresenta erros elevados tantos nos dados de treinamento quanto nos de validação e teste (Underfitting). Para resolver o problema, podemos aumentar a complexidade da rede, aumentar o número de épocas, ajustar os hiperparâmetros entre outros.


Já Overfitting, ocorre quando a topologia da rede é tão complexa, que a rede não aprende, mas sim, decora todos os dados de treinamento, inclusive ruídos e outras particularidades que não representam o problema de forma geral, por isso, tem dificuldade para generalizar para novos dados. Para a detecção, quando o modelo apresenta baixo erro no treinamento, mas erro significativamente grante na validação ou teste. Para resolver, podemos reduzir a complexidade da rede, aplicar técnicas de early stopping.